In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go

/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("data/real/heart.csv")

df


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [27]:
df_train = df_train.sample(frac=1).reset_index(drop=True)
df_train.value_counts('over_threshold')[0]
number_clients = 10

label_dict = {}
for i in df_train['over_threshold'].unique():
    num_instances = df_train['over_threshold'].value_counts().get(i)
    exp_instances = num_instances // number_clients
    label_dict[i] = exp_instances 

print(label_dict)
clients_dfs = [pd.DataFrame() for _ in range(number_clients)]
for i in range(number_clients):
    for j in label_dict.keys():
        temp_df = df_train[df_train['over_threshold'] == j].sample(n=label_dict[j])
        clients_dfs[i] = pd.concat([clients_dfs[i], temp_df], ignore_index=True)
        df_train = df_train.drop(temp_df.index)
    
    print(df_train.shape[0])
    if df_train.shape[0] < 0.1*num_instances:
        print("Last small set")
        clients_dfs[i] = pd.concat([clients_dfs[i], df_train], ignore_index=True)

#display(clients_dfs[2].value_counts('over_threshold'))

{np.int64(0): np.int64(2786), np.int64(1): np.int64(876)}
32969
29307
25645
21983
18321
14659
10997
7335
3673
11
Last small set


In [31]:
for index, obj in enumerate(clients_dfs):
    clients_dfs[index] = torch.tensor(obj.values)
print(type(clients_dfs[0]))

<class 'torch.Tensor'>


In [16]:
import torch

array = [[0,1,2,0,0],[3,4,5,6,7]]
tensor = torch.tensor(array)
tensor.size(1)

5